In [49]:
import sys
import os

PROJECT_ROOT = os.path.abspath("../..")
sys.path.append(PROJECT_ROOT)

import time
import pandas as pd
from Scripts.corss_validation.cross_val import cross_validate
from Scripts.models.random_forest import RandomForestModel

In [50]:
# load data
df = pd.read_csv("../../Data/pre_split_features.csv")
# dataset sampling for faster execution during development
# df = df.sample(n=4000,random_state=42).reset_index(drop=True)

In [51]:
param_grid = {
    "max_depth": [1, 2, 3, 4, 5],
}

n_outer = 10
n_inner = 3

In [52]:
start_time = time.time()
print("\n=== Base engineered features ===")
results = cross_validate(
    model_class=RandomForestModel,
    df=df,
    param_grid=param_grid,
    n_outer=n_outer,
    n_inner=n_inner,
    scoring="f1_macro",
    use_interactions=False
)
elapsed = time.time() - start_time
print(f"Execution time: {elapsed:.2f} sec")



=== Base engineered features ===
Samples: 94013, Features: 74
fold 1/10 f1_macro=0.3025 params={'max_depth': 2}
fold 2/10 f1_macro=0.3033 params={'max_depth': 2}
fold 3/10 f1_macro=0.3008 params={'max_depth': 3}
fold 4/10 f1_macro=0.3061 params={'max_depth': 2}
fold 5/10 f1_macro=0.3009 params={'max_depth': 2}
fold 6/10 f1_macro=0.2954 params={'max_depth': 2}
fold 7/10 f1_macro=0.3043 params={'max_depth': 2}
fold 8/10 f1_macro=0.3111 params={'max_depth': 3}
fold 9/10 f1_macro=0.3079 params={'max_depth': 2}
fold 10/10 f1_macro=0.3026 params={'max_depth': 2}

results:
  accuracy: 0.5143 +/- 0.0136
  precision_macro: 0.3273 +/- 0.0125
  recall_macro: 0.3047 +/- 0.0060
  f1_macro: 0.3035 +/- 0.0041
  train_f1: 0.3057 +/- 0.0021
  train_test_gap: 0.0022 +/- 0.0046
Execution time: 306.89 sec


In [53]:
start_time = time.time()

print("\n=== Enhanced behavioural features ===")
results_enhanced = cross_validate(
    model_class=RandomForestModel,
    df=df,
    param_grid=param_grid,
    n_outer=n_outer,
    n_inner=n_inner,
    scoring="f1_macro",
    use_interactions=True
)

elapsed = time.time() - start_time
print(f"Execution time: {elapsed:.2f} sec")


=== Enhanced behavioural features ===
Samples: 94013, Features: 100
fold 1/10 f1_macro=0.3091 params={'max_depth': 3}
fold 2/10 f1_macro=0.2982 params={'max_depth': 3}
fold 3/10 f1_macro=0.3007 params={'max_depth': 3}
fold 4/10 f1_macro=0.3022 params={'max_depth': 3}
fold 5/10 f1_macro=0.3068 params={'max_depth': 3}
fold 6/10 f1_macro=0.2969 params={'max_depth': 3}
fold 7/10 f1_macro=0.3077 params={'max_depth': 3}
fold 8/10 f1_macro=0.3142 params={'max_depth': 3}
fold 9/10 f1_macro=0.3142 params={'max_depth': 3}
fold 10/10 f1_macro=0.3016 params={'max_depth': 3}

results:
  accuracy: 0.4946 +/- 0.0071
  precision_macro: 0.3075 +/- 0.0089
  recall_macro: 0.3087 +/- 0.0061
  f1_macro: 0.3052 +/- 0.0059
  train_f1: 0.3070 +/- 0.0017
  train_test_gap: 0.0019 +/- 0.0056
Execution time: 473.52 sec


In [54]:
# save results
pd.DataFrame(results["fold_metrics"]).to_csv("../../Results/fold_metrics_rf.csv", index=False)
pd.DataFrame(results["best_params"]).to_csv("../../Results/best_params_rf.csv", index=False)

pd.DataFrame(results_enhanced["fold_metrics"]).to_csv("../../Results/fold_metrics_rf_enhanced.csv", index=False)
pd.DataFrame(results_enhanced["best_params"]).to_csv("../../Results/best_params_rf_enhanced.csv", index=False)

print("\nClass distribution:")
print(df["review_score"].value_counts(normalize=True))


Class distribution:
review_score
5    0.592535
4    0.197068
1    0.097114
3    0.082765
2    0.030517
Name: proportion, dtype: float64
